# Serveur Ollama sur un GPU gratuit (Colab ou Kaggle) pour le labo

**Kabyle LLM Benchmark, phase 2.** Quand le PC n'a pas assez de mémoire pour faire tourner Qwen, ce notebook le fait tourner sur le **GPU gratuit de Colab ou de Kaggle**. Le labo reste sur le PC : il prépare les questions (few-shot, RAG, contrôle anti-contamination) et envoie seulement la question finale à ce serveur, par un tunnel chiffré (Cloudflare).

## Ce qui part sur Colab

- **Uniquement les tests publics**, déjà publiés sur GitHub. Dès que l'adresse d'Ollama n'est plus celle du PC, le labo marque tous les modèles Ollama « tests publics seulement » et refuse tout run qui leur enverrait un test privé.
- Les exemples few-shot et RAG, tirés des corpus publics décontaminés.

## Étapes sur Kaggle (recommandé : tourne même onglet fermé)

1. Importer ce notebook (*File → Import Notebook*), puis à droite : **GPU T4 x2**, **Internet On**.
2. **Modèle fine-tuné (facultatif)** : *Add Input → Notebooks →* le notebook de fine-tuning terminé. Le `.gguf` est trouvé automatiquement.
3. *Save Version → Save & Run All (Commit)*. Dans les *Logs*, l'adresse `https://….trycloudflare.com` apparaît après quelques minutes.
4. Sur le PC, dans `.env` : `OLLAMA_DOCKER_URL=<cette adresse>`, puis `docker compose up -d api`.
5. À la fin des runs : arrêter la version (*Stop session*) pour ne pas consommer le quota GPU.

## Étapes sur Colab

1. *Exécution → Modifier le type d'exécution → T4 GPU*.
2. **Modèle fine-tuné (facultatif)** : renseigner `GGUF_ON_DRIVE` (chemin dans Mon Drive, voir `notebooks/README.md`).
3. *Exécution → Tout exécuter*, copier l'adresse affichée par la cellule du tunnel, puis comme l'étape 4 ci-dessus.
4. **Laisser l'onglet ouvert** pendant tout le run.

L'adresse du tunnel est aléatoire mais publique : ne la partager avec personne. Elle disparaît à la fin de la session.


In [ ]:
# ── Configuration ──────────────────────────────────────────
MODELS = ["qwen3.5:4b", "ministral-3:8b", "bge-m3"]   # mêmes modèles que le labo
GGUF_ON_DRIVE = ""                   # Colab : ex. "kab-llm/kab-qwen3.5-4b.Q4_K_M.gguf" (chemin dans Mon Drive)
# Modèles fine-tunés : nom attendu par models.yaml -> modèle de base (gabarit de conversation)
FINETUNED = {"kab-qwen3.5-4b": "qwen3.5:4b", "kab-ministral3-8b": "ministral-3:8b"}

import os, glob, shutil
# Colab a un dossier /kaggle et Kaggle a le module google.colab : marqueurs propres à chacun
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
IN_COLAB = not IN_KAGGLE and os.path.exists("/var/colab/hostname")
print("Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "autre machine")


In [ ]:
# ── Installer et démarrer Ollama ───────────────────────────
import os, subprocess, time, requests
SUDO = "sudo" if shutil.which("sudo") else ""  # Kaggle tourne déjà en root
!{SUDO} apt-get -qq install -y zstd > /dev/null
# GPU obligatoire : sans lui, Ministral 8B met 400 à 500 s par réponse
gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
print("=" * 60)
print("GPU :", gpu or "AUCUN : Settings → Accelerator → GPU T4 x2, puis relancer")
print("=" * 60)
# Téléchargement d'Ollama : GitHub renvoie parfois 504, on réessaie
for essai in range(1, 9):
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    if shutil.which("ollama"):
        break
    print(f"installation d'Ollama ratée (essai {essai}/8), nouvel essai dans 30 s")
    time.sleep(30)
assert shutil.which("ollama"), "Ollama non installé (serveur GitHub indisponible) : relancer dans quelques minutes"
env = dict(os.environ, OLLAMA_KEEP_ALIVE="24h", OLLAMA_MAX_LOADED_MODELS="2")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        print("Ollama", requests.get("http://127.0.0.1:11434/api/version", timeout=2).json()); break
    except Exception:
        time.sleep(1)

In [ ]:
# ── Télécharger les modèles ────────────────────────────────
for name in MODELS:
    !ollama pull {name}
for m in requests.get("http://127.0.0.1:11434/api/tags").json()["models"]:
    print(m["name"], m["digest"][:12])  # à comparer avec « ollama list » sur le PC

In [ ]:
# ── Modèles fine-tunés (facultatif) ───────────────────────
# Kaggle : sorties des notebooks de fine-tuning ajoutées en « Input ». Colab : fichier dans Google Drive.
def model_file(name):
    found = glob.glob(f"/kaggle/input/**/{name}*.gguf", recursive=True)
    found += glob.glob(f"/kaggle/input/**/{name}*/*.gguf", recursive=True)
    # le fichier du modèle de langue, pas celui de la partie vision (mmproj) éventuelle
    found = sorted({f for f in found if "mmproj" not in os.path.basename(f).lower()})
    print(name, ": fichiers .gguf trouvés :", found)
    return found[0] if found else None

ggufs = {}
if IN_KAGGLE:
    ggufs = {name: model_file(name) for name in FINETUNED}
elif GGUF_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ggufs = {"kab-qwen3.5-4b": f"/content/drive/MyDrive/{GGUF_ON_DRIVE}"}
    assert os.path.exists(ggufs["kab-qwen3.5-4b"]), f"introuvable : {GGUF_ON_DRIVE}"

for name, gguf in ggufs.items():
    if not gguf:
        continue
    print("Modèle fine-tuné :", name, "<-", gguf)
    # Même gabarit de conversation que son modèle de base : on reprend son Modelfile
    base = subprocess.run(["ollama", "show", FINETUNED[name], "--modelfile"], capture_output=True, text=True).stdout
    lines = [l for l in base.splitlines() if not l.startswith("FROM ")]
    open("Modelfile", "w").write(f"FROM {gguf}\n" + "\n".join(lines) + "\n")
    !ollama create {name} -f Modelfile
!ollama list


In [ ]:
# ── Préchauffage sur le GPU (vérifie que tout répond) ──────
r = requests.post("http://127.0.0.1:11434/api/chat", json={
    "model": MODELS[0], "messages": [{"role": "user", "content": "Bonjour"}],
    "stream": False, "think": False}, timeout=600)
print(r.json()["message"]["content"][:200])
!ollama ps

In [ ]:
# ── Tunnel Cloudflare : adresse à mettre dans .env ─────────
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:11434",
     "--http-host-header", "localhost:11434"],
    stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(120):
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if found:
        URL = found[0]; break
    time.sleep(1)
assert URL, "tunnel non démarré : voir cloudflared.log"
print("\n=== Dans .env sur le PC ===")
print(f"OLLAMA_DOCKER_URL={URL}")
print("puis : docker compose up -d api")


In [ ]:
# ── Garder la session active et suivre l'activité ──────────
# Laisser tourner pendant les runs (Colab : bouton ■ ; Kaggle : Stop session)
import datetime
while tunnel.poll() is None:
    ps = requests.get("http://127.0.0.1:11434/api/ps").json().get("models", [])
    print(datetime.datetime.now().strftime("%H:%M"), URL, "modèles chargés :", [m["name"] for m in ps], flush=True)
    time.sleep(300)
print("Le tunnel s'est arrêté : relancer le notebook")
